In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
from matplotlib import gridspec
import numpy as np
from matplotlib.patches import Patch
from sklearn.decomposition import PCA

os.chdir('/home/borisvdm/Documents/PhD/Lemonite/Lloyd-Price_IBD/results/LemonTree/')

# ---- CONFIGURATION ----
FOLDERNAME = 'ModuleViewer_files'
HEAT_FILE_EXPRESSION = 'LemonPreprocessed_expression.txt'  # For main expression heatmap
HEAT_FILE_COMPLETE = 'LemonPreprocessed_complete.txt'      # For regulator blocks
output_dir = 'Figures_with_scores'

# Option to display regulator scores in heatmap labels
SHOW_REGULATOR_SCORES = True

# Option to display PPI interactions on module figures
SHOW_PPIS = False  # Set to True to display protein-protein interactions on the left side of expression heatmaps

REGULATORS = [
    {"name": "Metabolites", "file": "Metabolite.percentile2_list.txt", "score_file": "Metabolite.percentile2.txt", "column": "metabolites"},
    {"name": "Transcription factors", "file": "Lovering.percentile2_list.txt", "score_file": "Lovering.percentile2.txt", "column": "genes"},
]

# Optional legend configurations
OPTIONAL_LEGENDS = {
    "origin": {
        "file": "sample_mapping_origin.mvf", 
        "title": "Sample Origin",
        "enabled": False
    }
}

# ---- HELPER FUNCTIONS ----
def load_sample_mapping(folder, filename):
    """Load sample mapping and legend from .mvf file"""
    with open(os.path.join(folder, filename), 'r') as f:
        lines = f.readlines()
    
    legend_line = [line for line in lines if line.startswith('::LEGEND=')][0]
    mapping_line = [line for line in lines if line.startswith('|')][0]
    
    legend_dict = {}
    legend_items = legend_line.split('=', 1)[1].split('\t')[0].split('|')
    for item in legend_items:
        label, color = item.split(':')
        legend_dict[label.strip()] = color.strip().lower()
    
    sample_color_dict = {}
    for item in mapping_line.strip('| \n').split('|'):
        sample, color = item.split(':')
        sample_color_dict[sample.strip()] = color.strip().lower()
    
    return legend_dict, sample_color_dict

def create_subset(main_df, module, cluster_data, gene_column='genes'):
    """Create ordered subset for a module"""
    mask = cluster_data['module'] == module
    if not mask.any():
        mask = cluster_data['module'].astype(str) == str(module)
    genes = cluster_data.loc[mask, gene_column].values[0]
    return main_df[main_df['symbol'].isin(genes)].set_index('symbol').loc[genes].reset_index()

def create_subset_with_scores(main_df, module, cluster_data, score_data=None, gene_column='genes', show_scores=False):
    """Create ordered subset for a module with optional scores"""
    mask = cluster_data['module'] == module
    if not mask.any():
        mask = cluster_data['module'].astype(str) == str(module)
    genes = cluster_data.loc[mask, gene_column].values[0]
    
    # Get the base subset
    subset = main_df[main_df['symbol'].isin(genes)].set_index('symbol').loc[genes].reset_index()
    
    # Add scores if score_data is provided and show_scores is True
    if score_data is not None and show_scores:
        # Filter score data for this module
        module_scores = score_data[score_data['Target'].astype(str) == str(module)]
        
        # Create a mapping of regulator to score
        score_mapping = dict(zip(module_scores['Regulator'], module_scores['Score']))
        
        # Update the symbol column to include scores
        new_symbols = []
        for symbol in subset['symbol']:
            if symbol in score_mapping:
                score = score_mapping[symbol]
                # Format score to 2 decimal places
                new_symbols.append(f"{symbol} ({score:.2f})")
            else:
                new_symbols.append(symbol)
        
        subset['symbol'] = new_symbols
    
    return subset

def load_metabolite_interactions(folder, filename):
    """Load metabolite-gene interactions from .mvf file"""
    metadata = {}
    data_rows = []
    file_path = os.path.join(folder, filename)
    
    if os.path.exists(file_path):
        with open(file_path, 'r') as f:
            for line in f:
                line = line.strip()
                if line.startswith('::'):
                    key_value = line[2:].split('=', 1) if '=' in line else line[2:].split(':', 1)
                    if len(key_value) == 2:
                        metadata[key_value[0].strip()] = key_value[1].strip()
                elif line and not line.startswith('#'):
                    data_rows.append(line.split('\t'))
        return pd.DataFrame(data_rows, columns=['Module', 'Genes', 'Metabolite']), metadata
    return pd.DataFrame(columns=['Module', 'Genes', 'Metabolite']), {}

def load_ppi_interactions(folder, filename):
    """Load PPI interactions from .mvf file"""
    metadata = {}
    data_rows = []
    file_path = os.path.join(folder, filename)
    
    if os.path.exists(file_path):
        with open(file_path, 'r') as f:
            for line in f:
                line = line.strip()
                if line.startswith('::'):
                    key_value = line[2:].split('=', 1) if '=' in line else line[2:].split(':', 1)
                    if len(key_value) == 2:
                        metadata[key_value[0].strip()] = key_value[1].strip()
                elif line and not line.startswith('#'):
                    parts = line.split('\t')
                    if len(parts) >= 2:
                        data_rows.append(parts)
        return pd.DataFrame(data_rows, columns=['Module', 'GenePair']), metadata
    return pd.DataFrame(columns=['Module', 'GenePair']), {}


# Create output dir
if not os.path.exists(output_dir):
    os.makedirs(output_dir)


# ---- DATA LOADING ----
# Load expression data for main heatmap
heat_expression = pd.read_csv(os.path.join(FOLDERNAME, '../Preprocessing/', HEAT_FILE_EXPRESSION), sep='\t')
# Load complete data for regulator blocks
heat_complete = pd.read_csv(os.path.join(FOLDERNAME, '../Preprocessing/', HEAT_FILE_COMPLETE), sep='\t')

print(f"Loaded expression data: {heat_expression.shape}")
print(f"Loaded complete data: {heat_complete.shape}")

cluster = pd.read_csv(os.path.join(FOLDERNAME, 'clusters_list.txt'), 
                     sep='\t', header=None, names=['module', 'genes'])
cluster['genes'] = cluster['genes'].str.split('|').apply(lambda x: pd.Categorical(x, categories=x, ordered=True))

# ---- FILTER MODULES BASED ON LemonTree_to_network RESULTS ----
specific_modules_file = './Networks/specific_modules.txt'
if os.path.exists(specific_modules_file):
    with open(specific_modules_file, 'r') as f:
        specific_modules = [line.strip() for line in f.readlines()]
    
    print(f"Found {len(specific_modules)} filtered modules from LemonTree_to_network")
    
    # Convert to same type as cluster module column for comparison
    if cluster['module'].dtype == 'object':
        specific_modules = [str(m) for m in specific_modules]
    else:
        specific_modules = [int(m) if m.isdigit() else m for m in specific_modules]
    
    # Filter cluster dataframe to only include modules that passed filtering
    original_module_count = len(cluster['module'].unique())
    cluster = cluster[cluster['module'].isin(specific_modules)]
    filtered_module_count = len(cluster['module'].unique())
    
    print(f"Modules before filtering: {original_module_count}")
    print(f"Modules after filtering: {filtered_module_count}")
    print(f"Removed {original_module_count - filtered_module_count} low-coherence modules")
    
else:
    print("Warning: specific_modules.txt not found - processing all modules")
    print("Make sure LemonTree_to_network.ipynb has been run successfully")

# Continue with the rest of your data loading...
loaded_regulators = []
for reg in REGULATORS:
    file_path = os.path.join(FOLDERNAME, reg['file'])
    if os.path.exists(file_path):
        df = pd.read_csv(file_path, sep='\t', header=None, names=['module', reg['column']])
        df[reg['column']] = df[reg['column']].str.split('|').apply(lambda x: pd.Categorical(x, categories=x, ordered=True))
        reg['data'] = df
        
        # Load score data if available
        score_file_path = os.path.join('Lemon_out', reg['score_file'])
        if os.path.exists(score_file_path):
            score_df = pd.read_csv(score_file_path, sep='\t')
            reg['score_data'] = score_df
            if SHOW_REGULATOR_SCORES:
                print(f"Loaded scores for {reg['name']} from {reg['score_file']}")
        else:
            reg['score_data'] = None
            if SHOW_REGULATOR_SCORES:
                print(f"Warning: Score file {reg['score_file']} not found for {reg['name']}")
        
        loaded_regulators.append(reg)

# Load sample mappings and metabolite interactions
legend_dict, sample_color_dict = load_sample_mapping(FOLDERNAME, 'sample_mapping.mvf')

# Load optional legends based on configuration
loaded_legends = []
all_colors = set(legend_dict.values())

# Main legend is always included
loaded_legends.append({
    "legend": legend_dict,
    "samples": sample_color_dict,
    "title": "Clinical Status"
})

# Load optional legends if enabled and files exist
for key, config in OPTIONAL_LEGENDS.items():
    if config["enabled"]:
        file_path = os.path.join(FOLDERNAME, config["file"])
        if os.path.exists(file_path):
            try:
                legend, samples = load_sample_mapping(FOLDERNAME, config["file"])
                loaded_legends.append({
                    "legend": legend,
                    "samples": samples,
                    "title": config["title"]
                })
                all_colors.update(legend.values())
                print(f"Loaded optional legend: {config['title']}")
            except Exception as e:
                print(f"Warning: Could not load {config['title']} from {config['file']}: {str(e)}")
        else:
            print(f"Warning: Optional legend file {config['file']} not found, skipping {config['title']}")

# Combine color mappings
color_mapping = {color: color.lower() for color in all_colors}

# Create legend elements for all loaded legends
legend_elements = []
for legend_data in loaded_legends:
    elements = [Patch(facecolor=color_mapping[c], label=l, edgecolor='black', linewidth=0.5) 
                for l, c in legend_data["legend"].items()]
    legend_elements.append((elements, legend_data["title"]))

# Load metabolite interactions
metabo, meta_metadata = load_metabolite_interactions(FOLDERNAME, 'metabolite_LemonIteKG_interactions.mvf')

# Load PPI interactions
ppis, ppi_metadata = load_ppi_interactions(FOLDERNAME, 'PPI_interactions.mvf')
if not ppis.empty:
    print(f"Loaded {len(ppis)} PPI interactions")
else:
    print("No PPI interactions file found or file is empty")

# ---- MAIN PROCESSING ----
results_dir = os.path.join(output_dir)
os.makedirs(results_dir, exist_ok=True)

for module_number in cluster['module'].unique():
    try:
        print(f"\nProcessing module {module_number}...")
        
        # Create eigengene using expression data
        subset_cluster = create_subset(heat_expression, module_number, cluster)
        subset_cluster = subset_cluster.sort_values('symbol')
        module_expr = subset_cluster.iloc[:, 2:]
        pca = PCA(n_components=1)
        eigengene = pca.fit_transform(module_expr.T)
        eigengene_series = pd.Series(eigengene.flatten(), index=module_expr.columns, name='Eigengene')
        sorted_samples = eigengene_series.sort_values().index.tolist()
        
        # Prepare subsets for all regulators using complete data
        subsets = []
        titles = []
        for reg in loaded_regulators:
            try:
                subset = create_subset_with_scores(heat_complete, module_number, reg['data'], 
                                                 score_data=reg.get('score_data'), 
                                                 gene_column=reg['column'],
                                                 show_scores=SHOW_REGULATOR_SCORES)
                if not subset.empty:
                    # Scale transcription factor data (other data types are pre-scaled)
                    if reg['name'] == 'Transcription factors':
                        # Z-score normalization across samples for each gene
                        numeric_cols = subset.columns[2:]  # Skip 'symbol' and first column
                        subset[numeric_cols] = subset[numeric_cols].apply(
                            lambda row: (row - row.mean()) / row.std() if row.std() > 0 else row - row.mean(), 
                            axis=1
                        )
                        print(f"  Scaled {reg['name']} data using z-score normalization")
                    
                    subsets.append(subset)
                    titles.append(reg['name'])
            except Exception as e:
                print(f" Error loading {reg['name']}: {str(e)}")
                continue

        # Add main expression data
        subsets.append(subset_cluster)
        titles.append("Expression Data")

        # Create figure with extended grid
        total_rows = sum(len(s) for s in subsets)
        num_legends = len(loaded_legends)
        fig = plt.figure(figsize=(15, total_rows/2 + 8))
        fig.suptitle(f'Module {module_number}', fontsize=18, fontweight='bold', y=0.92)
        
        # Grid layout: dynamic rows for annotations + legends
        gs = gridspec.GridSpec(len(subsets) + num_legends + 3, 2,
                             height_ratios=[len(s) for s in subsets] + [1] * num_legends + [1.5, 1.5, 1.5],
                             width_ratios=[1, 1],
                             hspace=0.25, wspace=0.05)

        # Plot each heatmap
        for idx, (subset_df, title) in enumerate(zip(subsets, titles)):
            ax = fig.add_subplot(gs[idx, :])
            numeric_df = subset_df.iloc[:, 2:].reindex(columns=sorted_samples)
            
            # All subsets are created from the main expression dataframe (heat)
            # so they all have the 'symbol' column with gene symbols
            labels = subset_df['symbol'].values

            sns.heatmap(numeric_df,
                       cmap=LinearSegmentedColormap.from_list('custom', ['blue', 'black', 'yellow']),
                       vmin=-2.0, vmax=2.0,
                       yticklabels=labels,
                       xticklabels=False,
                       cbar=False,
                       ax=ax,
                       linewidths=0.5,
                       linecolor='black')
            
            # Calculate optimal font size based on row height
            ax_height_inches = ax.get_position().height * fig.get_figheight()
            row_height_points = (ax_height_inches * 72) / len(labels)
            optimal_fontsize = max(6, min(row_height_points * 0.9, 14))
            
            ax.set_xticklabels(ax.get_xticklabels(), rotation=90, fontsize=6)
            ax.yaxis.tick_right()
            ax.yaxis.set_label_position('right')
            ax.set_yticklabels(labels, rotation=0, va='center', fontsize=optimal_fontsize, fontweight='bold')
            
            # Dynamic title positioning and sizing based on heatmap size
            num_genes = len(labels)
            
            if num_genes <= 5:
                title_fontsize = 14
                dynamic_pad = 8
            elif num_genes <= 15:
                title_fontsize = 16
                dynamic_pad = 12
            else:
                title_fontsize = 18
                dynamic_pad = 20
            
            # Add PPI interactions for expression data (on the left side)
            if title == "Expression Data" and SHOW_PPIS and not ppis.empty:
                module_ppis = ppis[ppis['Module'] == str(module_number)]
                
                if len(module_ppis) > 0:
                    # Get the position of the main heatmap
                    pos = ax.get_position()
                    
                    # Create a new axis on the left side for PPI connections
                    ppi_width = 0.03
                    ax_ppi = fig.add_axes([
                        pos.x0 - ppi_width - 0.01,
                        pos.y0,
                        ppi_width,
                        pos.height
                    ])
                    
                    ax_ppi.set_xlim(0, 1)
                    ax_ppi.set_ylim(0, len(labels))
                    ax_ppi.axis('off')
                    
                    # Get PPI color from metadata (default is blue)
                    ppi_color = ppi_metadata.get('COLOR', 'blue').lower()
                    
                    # Draw lines connecting genes that have PPIs
                    for _, ppi_row in module_ppis.iterrows():
                        gene_pair = ppi_row['GenePair'].split('|')
                        if len(gene_pair) == 2:
                            gene1, gene2 = gene_pair[0].strip(), gene_pair[1].strip()
                            
                            try:
                                idx1 = list(labels).index(gene1)
                                idx2 = list(labels).index(gene2)
                                
                                y1 = len(labels) - idx1 - 0.5
                                y2 = len(labels) - idx2 - 0.5
                                
                                ax_ppi.plot([0.2, 0.8], [y1, y2], 
                                          color=ppi_color, 
                                          linewidth=1.5, 
                                          alpha=0.7,
                                          solid_capstyle='round')
                                
                                ax_ppi.plot([0.2], [y1], 'o', 
                                          color=ppi_color, 
                                          markersize=3, 
                                          alpha=0.8)
                                ax_ppi.plot([0.8], [y2], 'o', 
                                          color=ppi_color, 
                                          markersize=3, 
                                          alpha=0.8)
                                
                            except ValueError:
                                continue
                    
                    print(f"  Added {len(module_ppis)} PPI connections to visualization")

            # Add metabolite interactions for expression data
            if title == "Expression Data":
                module_data = metabo[metabo['Module'] == str(module_number)]
                metabolites = module_data['Metabolite'].unique()
                
                if len(metabolites) > 0:
                    match_matrix = np.zeros((len(labels), len(metabolites)))
                    for i, gene in enumerate(labels):
                        for j, metabolite in enumerate(metabolites):
                            gene_lists = module_data[module_data['Metabolite'] == metabolite]['Genes'].values
                            if any(gene in gl.split('|') for gl in gene_lists):
                                match_matrix[i, j] = 1

                    meta_color = meta_metadata.get('COLOR', 'yellow').lower()
                    pos = ax.get_position()
                    num_cols_main = numeric_df.shape[1]
                    cell_width_main = pos.width / num_cols_main
                    metabolite_panel_width = cell_width_main * len(metabolites)

                    ax_meta = fig.add_axes([
                        pos.x1 + 0.10,
                        pos.y0,
                        metabolite_panel_width,
                        pos.height
                    ])

                    sns.heatmap(
                        match_matrix,
                        cmap=LinearSegmentedColormap.from_list('custom_meta', ['white', meta_color]),
                        cbar=False,
                        xticklabels=metabolites,
                        yticklabels=False,
                        linewidths=0.5,
                        linecolor='black',
                        ax=ax_meta,
                        square=False,
                        clip_on=False
                    )
                    ax_meta.set_xticklabels(metabolites, rotation=90, fontsize=10, fontweight='bold')
        
        # Annotation bars - dynamically create based on loaded legends
        for i, legend_data in enumerate(loaded_legends):
            color_data = [color_mapping[legend_data["samples"].get(s, 'grey')] for s in sorted_samples]
            
            ax_anno = fig.add_subplot(gs[len(subsets) + i, :])
            for j, color in enumerate(color_data):
                rect = plt.Rectangle((j, 0), 1, 1, facecolor=color, edgecolor='black', linewidth=0.5)
                ax_anno.add_patch(rect)
            ax_anno.set_xlim(0, len(color_data))
            ax_anno.set_ylim(0, 1)
            ax_anno.axis('off')
            
            # Add title to the left of the annotation bar
            ax_anno.text(-0.01, 0.5, legend_data["title"], 
                        transform=ax_anno.transAxes,
                        rotation=0,
                        va='center', 
                        ha='right',
                        fontsize=10,
                        fontweight='bold')

        # Legends - dynamically create based on loaded legends
        for i, (elements, title) in enumerate(legend_elements):
            ax_leg = fig.add_subplot(gs[len(subsets) + num_legends + i, 0])
            ax_leg.axis('off')
            ax_leg.legend(handles=elements, loc='center left', 
                        frameon=False, fontsize=10, title=title,
                        bbox_to_anchor=(0, 0.5))

        # Color bar
        ax_colorbar = fig.add_subplot(gs[len(subsets) + num_legends:, 1])
        ax_colorbar.axis('off')
        cax = fig.add_axes([0.4, 0.12, 0.4, 0.03])
        cbar = plt.colorbar(plt.cm.ScalarMappable(
            norm=plt.Normalize(-2.0, 2.0),
            cmap=LinearSegmentedColormap.from_list('custom', ['blue', 'black', 'yellow'])
        ), cax=cax, orientation='horizontal')
        cbar.set_ticks([-2.0, 0, 2.0])
        cbar.set_ticklabels(['<= -2.0', '0', '>= 2.0'])
        cbar.outline.set_linewidth(0.5)
        cbar.ax.set_title('Expression ratios', fontsize=12, pad=15)

        plt.savefig(f'./{output_dir}/Module_{module_number}_heatmap.png', 
                   dpi=600, bbox_inches='tight')
        plt.savefig(f'./{output_dir}/Module_{module_number}_heatmap.pdf', 
                   dpi=600, bbox_inches='tight')
        plt.close()
        print(f"Saved module {module_number}")
        
        
    except Exception as e:
        print(f"Error in module {module_number}: {str(e)}")
        import traceback
        plt.close('all')
        traceback.print_exc()

Loaded expression data: (5291, 77)
Loaded complete data: (5762, 77)
Found 86 filtered modules from LemonTree_to_network
Modules before filtering: 97
Modules after filtering: 86
Removed 11 low-coherence modules
Loaded scores for Metabolites from Metabolite.percentile2.txt
Loaded scores for Transcription factors from Lovering.percentile2.txt
Loaded 3792 PPI interactions

Processing module 0...
  Scaled Transcription factors data using z-score normalization
Saved module 0

Processing module 1...
 Error loading Metabolites: "['C22_5_CE_[M_NH4]_'] not in index"
  Scaled Transcription factors data using z-score normalization
Saved module 1

Processing module 2...
  Scaled Transcription factors data using z-score normalization
Saved module 2

Processing module 3...
  Scaled Transcription factors data using z-score normalization
Saved module 3

Processing module 4...
 Error loading Metabolites: "['C18_0_CE_[M_NH4]_', 'C38_4_DAG_[M_NH4]_'] not in index"
  Scaled Transcription factors data using

Traceback (most recent call last):
  File "/tmp/ipykernel_380686/1243505533.py", line 272, in <module>
    subset_cluster = create_subset(heat_expression, module_number, cluster)
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_380686/1243505533.py", line 67, in create_subset
    return main_df[main_df['symbol'].isin(genes)].set_index('symbol').loc[genes].reset_index()
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^
  File "/home/borisvdm/Software/miniconda3/envs/LemonIte/lib/python3.11/site-packages/pandas/core/indexing.py", line 1191, in __getitem__
    return self._getitem_axis(maybe_callable, axis=axis)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/borisvdm/Software/miniconda3/envs/LemonIte/lib/python3.11/site-packages/pandas/core/indexing.py", line 1420, in _getitem_axis
    return self._getitem_iterable(key, axis=axis)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  Fil

Saved module 17

Processing module 18...
  Scaled Transcription factors data using z-score normalization
Saved module 18

Processing module 19...
Error in module 19: "['PIFO'] not in index"

Processing module 20...


Traceback (most recent call last):
  File "/tmp/ipykernel_380686/1243505533.py", line 272, in <module>
    subset_cluster = create_subset(heat_expression, module_number, cluster)
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_380686/1243505533.py", line 67, in create_subset
    return main_df[main_df['symbol'].isin(genes)].set_index('symbol').loc[genes].reset_index()
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^
  File "/home/borisvdm/Software/miniconda3/envs/LemonIte/lib/python3.11/site-packages/pandas/core/indexing.py", line 1191, in __getitem__
    return self._getitem_axis(maybe_callable, axis=axis)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/borisvdm/Software/miniconda3/envs/LemonIte/lib/python3.11/site-packages/pandas/core/indexing.py", line 1420, in _getitem_axis
    return self._getitem_iterable(key, axis=axis)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  Fil

  Scaled Transcription factors data using z-score normalization
Saved module 20

Processing module 21...
  Scaled Transcription factors data using z-score normalization
Saved module 21

Processing module 22...
 Error loading Metabolites: "['C44_0_TAG_[M_NH4]_'] not in index"
  Scaled Transcription factors data using z-score normalization
Saved module 22

Processing module 23...
 Error loading Metabolites: "['C50_0_TAG_[M_NH4]_'] not in index"
  Scaled Transcription factors data using z-score normalization
Saved module 23

Processing module 24...
  Scaled Transcription factors data using z-score normalization
Saved module 24

Processing module 25...
Error in module 25: "['CCL3L1'] not in index"

Processing module 26...
  Scaled Transcription factors data using z-score normalization


...

Saved module 60

Processing module 61...
Error in module 61: "['RNF165'] not in index"

Processing module 62...
 Error loading Metabolites: "['C22_5_CE_[M_NH4]_'] not in index"
  Scaled Transcription factors data using z-score normalization


Traceback (most recent call last):
  File "/tmp/ipykernel_380686/1243505533.py", line 272, in <module>
    subset_cluster = create_subset(heat_expression, module_number, cluster)
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_380686/1243505533.py", line 67, in create_subset
    return main_df[main_df['symbol'].isin(genes)].set_index('symbol').loc[genes].reset_index()
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^
  File "/home/borisvdm/Software/miniconda3/envs/LemonIte/lib/python3.11/site-packages/pandas/core/indexing.py", line 1191, in __getitem__
    return self._getitem_axis(maybe_callable, axis=axis)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/borisvdm/Software/miniconda3/envs/LemonIte/lib/python3.11/site-packages/pandas/core/indexing.py", line 1420, in _getitem_axis
    return self._getitem_iterable(key, axis=axis)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  Fil

Saved module 62

Processing module 63...
  Scaled Transcription factors data using z-score normalization
Saved module 63

Processing module 64...
  Scaled Transcription factors data using z-score normalization
Saved module 64

Processing module 66...
  Scaled Transcription factors data using z-score normalization
Saved module 66

Processing module 67...
 Error loading Metabolites: "['C48_1_TAG_[M_NH4]_'] not in index"
  Scaled Transcription factors data using z-score normalization
Saved module 67

Processing module 68...
 Error loading Metabolites: "['C50_2_TAG_[M_NH4]_'] not in index"
  Scaled Transcription factors data using z-score normalization
Saved module 68

Processing module 69...
 Error loading Metabolites: "['C16_1_MAG_[M_NH4]_', 'C42_0_TAG_[M_NH4]_'] not in index"
  Scaled Transcription factors data using z-score normalization
Saved module 69

Processing module 70...
  Scaled Transcription factors data using z-score normalization
Saved module 70

Processing module 71...
  Sc

Traceback (most recent call last):
  File "/tmp/ipykernel_380686/1243505533.py", line 272, in <module>
    subset_cluster = create_subset(heat_expression, module_number, cluster)
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_380686/1243505533.py", line 67, in create_subset
    return main_df[main_df['symbol'].isin(genes)].set_index('symbol').loc[genes].reset_index()
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^
  File "/home/borisvdm/Software/miniconda3/envs/LemonIte/lib/python3.11/site-packages/pandas/core/indexing.py", line 1191, in __getitem__
    return self._getitem_axis(maybe_callable, axis=axis)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/borisvdm/Software/miniconda3/envs/LemonIte/lib/python3.11/site-packages/pandas/core/indexing.py", line 1420, in _getitem_axis
    return self._getitem_iterable(key, axis=axis)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  Fil

Saved module 78

Processing module 79...
  Scaled Transcription factors data using z-score normalization
Saved module 79

Processing module 81...
 Error loading Metabolites: "['C42_0_TAG_[M_NH4]_'] not in index"
  Scaled Transcription factors data using z-score normalization
Saved module 81

Processing module 82...
  Scaled Transcription factors data using z-score normalization
Saved module 82

Processing module 83...
 Error loading Metabolites: "['C52_0_TAG_[M_NH4]_', 'C46_0_TAG_[M_NH4]_'] not in index"
  Scaled Transcription factors data using z-score normalization
Saved module 83

Processing module 84...
  Scaled Transcription factors data using z-score normalization
Saved module 84

Processing module 85...
  Scaled Transcription factors data using z-score normalization
Saved module 85

Processing module 86...
 Error loading Metabolites: "['C46_0_TAG_[M_NH4]_'] not in index"
  Scaled Transcription factors data using z-score normalization
Saved module 86

Processing module 91...
  Sc